# Parkinson's Disease Voice Screening & Clinical Decision Support Platform
## Notebook 03: Feature Extraction via Frozen WavLM-Base-Plus & Feature Caching

> **DISCLAIMER:** This software is a research screening tool, **NOT** a diagnostic device.

### Purpose & Architecture Alignment
According to [docs/DECISIONS.md](file:///Users/eshwarsaielugam/Documents/prototype/docs/DECISIONS.md), upstream self-supervised acoustic representations are extracted using **`microsoft/wavlm-base-plus`** (MIT license).
- **Frozen Feature Extractor:** WavLM is kept completely frozen (no fine-tuning / `requires_grad=False`) to leverage rich speech-denoised acoustic priors without catastrophic forgetting or overfitting on small clinical cohorts.
- **Disk Feature Caching:** Recomputing 12-layer Transformer forward passes every training epoch would quickly exhaust Colab's free-tier GPU quotas. Instead, this notebook performs a single forward pass over all preprocessed recordings (16kHz mono, 4.0s windows) and caches the resulting sequence embeddings as `.npy` files.
- **Dimensionality Contract:** Each processed 4.0-second ($64{,}000$ sample) clip produces a fixed-size temporal sequence tensor of shape **$(T=199, D=768)$**. This exact $T=199$ dimension is passed forward to configure the ConvNeXt V2 convolutional adapter and Transformer Encoder in subsequent stages.
- **Resumability:** The extraction pipeline is fully idempotent—it checks for the existence of valid `.npy` files before computing, allowing seamless resumption across Colab runtime timeouts or disconnections.


In [1]:
# Cell 1: Install & import required libraries
import sys
import subprocess

required_packages = ["torch", "torchaudio", "transformers", "soundfile", "pandas", "numpy", "tqdm"]
for pkg in required_packages:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os
from pathlib import Path
import time
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from tqdm import tqdm
from transformers import Wav2Vec2FeatureExtractor, WavLMModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("Device Name:     Apple Silicon MPS")
else:
    print("Device Name:     CPU")
print("All dependencies successfully imported.")


PyTorch version: 2.8.0
CUDA available:  False
Device Name:     Apple Silicon MPS
All dependencies successfully imported.


In [2]:
# Cell 2: Load microsoft/wavlm-base-plus, freeze parameters, and determine sequence length T
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata.csv"
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Select hardware device
if torch.cuda.is_available():
    device = torch.device("cuda")
    autocast_device = "cuda"
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    autocast_device = None
else:
    device = torch.device("cpu")
    autocast_device = None

print(f"Active compute device: {device}")

# Load model and feature extractor
MODEL_ID = "microsoft/wavlm-base-plus"
print(f"Loading pretrained model: {MODEL_ID}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
model = WavLMModel.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

# Explicitly freeze all parameters
model.requires_grad_(False)

# Strict assertion: verify no gradients are tracked
unfrozen_params = [name for name, p in model.named_parameters() if p.requires_grad]
assert len(unfrozen_params) == 0, f"Found unfrozen parameters: {unfrozen_params}"
print("✓ Strict assertion verified: All WavLM parameters are completely frozen (0 trainable parameters).")

# Determine exact empirical sequence length T for 4.0s (64,000 samples @ 16kHz)
DUMMY_SAMPLES = 64000
dummy_audio = torch.zeros(1, DUMMY_SAMPLES, device=device)
with torch.no_grad():
    if autocast_device:
        with torch.autocast(device_type=autocast_device, dtype=torch.float16):
            dummy_out = model(dummy_audio).last_hidden_state
    else:
        dummy_out = model(dummy_audio).last_hidden_state

EXACT_T = dummy_out.shape[1]
EXACT_DIM = dummy_out.shape[2]

print("\n=== FIXED FEATURE DIMENSION CONTRACT ===")
print(f"  Input Window:       {DUMMY_SAMPLES} samples (4.0s @ 16,000 Hz)")
print(f"  Sequence Length T:  {EXACT_T} frames")
print(f"  Embedding Dim D:    {EXACT_DIM} features")
print(f"  Output Tensor:      (batch_size, {EXACT_T}, {EXACT_DIM})")
print("==========================================")


Active compute device: mps
Loading pretrained model: microsoft/wavlm-base-plus...
✓ Strict assertion verified: All WavLM parameters are completely frozen (0 trainable parameters).

=== FIXED FEATURE DIMENSION CONTRACT ===
  Input Window:       64000 samples (4.0s @ 16,000 Hz)
  Sequence Length T:  199 frames
  Embedding Dim D:    768 features
  Output Tensor:      (batch_size, 199, 768)


In [3]:
# Cell 3: Batched feature extraction loop with resumability and autocast
df = pd.read_csv(METADATA_PATH)
print(f"Total recordings in metadata: {len(df)}")

BATCH_SIZE = 16
total_extracted = 0
total_skipped = 0

t_start = time.time()

for split in ["train", "val", "test"]:
    split_df = df[df["split"] == split].copy()
    print(f"\n--- Extracting Features for Split: {split.upper()} ({len(split_df)} files) ---")

    batch_audio = []
    batch_dest_paths = []

    for idx, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Split {split}"):
        proc_path = PROJECT_ROOT / row["processed_path"]
        subject_id = row["subject_id"]
        stem = proc_path.stem

        # Destination path: data/processed/features/{split}/{subject_id}/{stem}.npy
        dest_dir = FEATURES_DIR / split / subject_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest_file = dest_dir / f"{stem}.npy"

        # Resumable skip check: verify file exists and matches exact shape
        if dest_file.exists():
            try:
                cached = np.load(str(dest_file))
                if cached.shape == (EXACT_T, EXACT_DIM):
                    total_skipped += 1
                    continue
            except Exception:
                pass

        audio, sr = sf.read(str(proc_path))
        batch_audio.append(audio)
        batch_dest_paths.append(dest_file)

        if len(batch_audio) == BATCH_SIZE:
            tensor_in = torch.tensor(np.array(batch_audio), dtype=torch.float32, device=device)
            with torch.no_grad():
                if autocast_device:
                    with torch.autocast(device_type=autocast_device, dtype=torch.float16):
                        feats = model(tensor_in).last_hidden_state
                else:
                    feats = model(tensor_in).last_hidden_state

            feats_np = feats.cpu().to(torch.float32).numpy()
            for arr, p in zip(feats_np, batch_dest_paths):
                np.save(str(p), arr)
                total_extracted += 1

            batch_audio.clear()
            batch_dest_paths.clear()

    # Flush remainder of split
    if batch_audio:
        tensor_in = torch.tensor(np.array(batch_audio), dtype=torch.float32, device=device)
        with torch.no_grad():
            if autocast_device:
                with torch.autocast(device_type=autocast_device, dtype=torch.float16):
                    feats = model(tensor_in).last_hidden_state
            else:
                feats = model(tensor_in).last_hidden_state

        feats_np = feats.cpu().to(torch.float32).numpy()
        for arr, p in zip(feats_np, batch_dest_paths):
            np.save(str(p), arr)
            total_extracted += 1

        batch_audio.clear()
        batch_dest_paths.clear()

elapsed = time.time() - t_start
print(f"\nExtraction summary: {total_extracted} newly extracted, {total_skipped} skipped (already cached).")
print(f"Elapsed time: {elapsed:.2f} seconds.")

# Record relative feature paths in metadata
df["feature_path"] = [
    str((FEATURES_DIR / row["split"] / row["subject_id"] / f"{Path(row['processed_path']).stem}.npy").relative_to(PROJECT_ROOT))
    for _, row in df.iterrows()
]
df.to_csv(METADATA_PATH, index=False)
print(f"Updated metadata table written to: {METADATA_PATH}")


Total recordings in metadata: 831

--- Extracting Features for Split: TRAIN (584 files) ---
Split train: 100%|██████████| 584/584 [01:51<00:00,  5.23it/s]

--- Extracting Features for Split: VAL (139 files) ---
Split val: 100%|██████████| 139/139 [00:26<00:00,  5.29it/s]

--- Extracting Features for Split: TEST (108 files) ---
Split test: 100%|██████████| 108/108 [00:20<00:00,  5.31it/s]

Extraction summary: 831 newly extracted, 0 skipped (already cached).
Elapsed time: 138.45 seconds.
Updated metadata table written to: /Users/eshwarsaielugam/Documents/prototype/data/processed/metadata.csv


In [4]:
# Cell 4: Integrity audit assertions
print("=== FEATURE CACHE AUDIT & CONTRACT VERIFICATION ===")

assert "feature_path" in df.columns, "Column 'feature_path' missing from metadata.csv"
assert df["feature_path"].isna().sum() == 0, "Found null feature paths in metadata.csv"
print(f"✓ Metadata integrity verified: {len(df)} rows contain valid feature_path entries.")

shapes = []
dtypes = []
corrupt = []

for p_rel in tqdm(df["feature_path"], desc="Validating Feature Tensors"):
    full_path = PROJECT_ROOT / p_rel
    if not full_path.exists():
        corrupt.append((p_rel, "File does not exist"))
        continue
    try:
        arr = np.load(str(full_path))
        shapes.append(arr.shape)
        dtypes.append(arr.dtype)
    except Exception as exc:
        corrupt.append((p_rel, str(exc)))

assert len(corrupt) == 0, f"Found {len(corrupt)} missing or corrupt feature files: {corrupt[:5]}"
print(f"✓ File presence verified: All {len(df)} .npy feature files exist and load successfully.")

unique_shapes = set(shapes)
assert unique_shapes == {(EXACT_T, EXACT_DIM)}, f"Unexpected tensor shapes detected: {unique_shapes}"
print(f"✓ Shape uniformity verified: All tensors are exactly ({EXACT_T}, {EXACT_DIM}).")

unique_dtypes = set(dtypes)
assert unique_dtypes == {np.dtype("float32")}, f"Unexpected dtypes detected: {unique_dtypes}"
print(f"✓ Precision verified: All tensors are float32.")

# Resumability check: verify that a second pass skips all 831 files
already_cached = sum(
    1 for p in df["feature_path"]
    if (PROJECT_ROOT / p).exists() and np.load(str(PROJECT_ROOT / p)).shape == (EXACT_T, EXACT_DIM)
)
assert already_cached == len(df), f"Expected {len(df)} cached files, found {already_cached}"
print(f"✓ Resumability verified: 100% of files ({already_cached}/{len(df)}) are idempotent and cached.")

print(f"\nSTATUS: ALL FEATURE EXTRACTION CHECKS PASSED.")
print(f"KNOWN DOWNSTREAM DIMENSION CONTRACT: Input shape to ConvNeXt V2 adapter is ({EXACT_T}, {EXACT_DIM}).")


=== FEATURE CACHE AUDIT & CONTRACT VERIFICATION ===
✓ Metadata integrity verified: 831 rows contain valid feature_path entries.
Validating Feature Tensors: 100%|██████████| 831/831 [00:00<00:00, 1421.18it/s]
✓ File presence verified: All 831 .npy feature files exist and load successfully.
✓ Shape uniformity verified: All tensors are exactly (199, 768).
✓ Precision verified: All tensors are float32.
✓ Resumability verified: 100% of files (831/831) are idempotent and cached.

STATUS: ALL FEATURE EXTRACTION CHECKS PASSED.
KNOWN DOWNSTREAM DIMENSION CONTRACT: Input shape to ConvNeXt V2 adapter is (199, 768).
